In [20]:
import os
import numpy as np
import pandas as pd
from pathlib import Path


# Same constant used in calcpath.py
c = 0.0814514


def flow_observables_from_y(y):
    """
    Reproduce the observables used by the driver from an evolved
    Hubble-flow state y.

    Expected indexing:
        y[0] = phi
        y[1] = H
        y[2] = epsilon
        y[3] = sigma
        y[4] = lambda_2
        y[5] = lambda_3
        y[6] = lambda_4
        y[7] = lambda_5
    """

    eps = y[2]
    sig = y[3]
    lam2 = y[4]
    lam3 = y[5]

    # -----------------------------------------
    # r
    # Same as tsratio(y)
    # -----------------------------------------
    r = 16.0 * eps * (
        1.0 - c * (sig + 2.0 * eps)
    )

    # -----------------------------------------
    # n_s
    # Same as specindex(y)
    # -----------------------------------------
    ns = (
        1.0
        + sig
        - (5.0 - 3.0*c) * eps**2
        - 0.25 * (3.0 - 5.0*c) * eps * sig
        + 0.5 * (3.0 - c) * lam2
    )

    # -----------------------------------------
    # Flow derivatives needed for alpha_s
    # Same equations as derivs()
    # -----------------------------------------
    deps_dN = eps * (sig + 2.0*eps)

    dsig_dN = (
        2.0*lam2
        - 5.0*eps*sig
        - 12.0*eps**2
    )

    # For i = 4 in your flow hierarchy:
    #
    # d(lambda_2)/dN
    # = [0.5*(4-3)*sigma + (4-4)*epsilon]*lambda_2
    #   + lambda_3
    #
    # = 0.5*sigma*lambda_2 + lambda_3
    dlam2_dN = 0.5 * sig * lam2 + lam3

    # -----------------------------------------
    # alpha_s
    # Same as dspecindex(y)
    # -----------------------------------------
    alpha_s = -(
        1.0 / (1.0 - eps)
    ) * (
        dsig_dN

        - 2.0*(5.0 - 3.0*c)
        * eps
        * deps_dN

        - 0.25*(3.0 - 5.0*c)
        * (
            eps*dsig_dN
            + sig*deps_dN
        )

        + 0.5*(3.0 - c)
        * dlam2_dN
    )

    return {
        "r_from_esigma": r,
        "ns_from_esigma": ns,
        "alpha_s_from_esigma": alpha_s,

        "eps_N60": eps,
        "sig_N60": sig,
        "lam2_N60": lam2,
        "lam3_N60": lam3,
        "lam4_N60": y[6],
        "lam5_N60": y[7],
    }

To compare every accepted model in one summary file:

In [21]:
def check_esigma_against_summary(
    summary_file,
    neqs=8,
):
    """
    For every accepted model in summary_file:

    1. Read test_esigma_neqsX.dat from its outdir.
    2. Recompute r, n_s, alpha_s from the evolved y state.
    3. Compare against the values saved in the summary.
    """

    summary = pd.read_csv(summary_file)

    rows = []

    for _, model in summary.iterrows():

        outdir = model["outdir"]

        esigma_file = os.path.join(
            outdir,
            f"test_esigma_neqs{neqs}.dat"
        )

        if not os.path.exists(esigma_file):
            print(
                f"WARNING: missing file:\n{esigma_file}"
            )
            continue

        # File contains:
        # phi H eps sig lam2 lam3 lam4 lam5 Nefolds
        data = np.loadtxt(esigma_file)

        if data.ndim > 1:
            data = data.ravel()

        if len(data) < neqs + 1:
            raise ValueError(
                f"{esigma_file} has only {len(data)} values; "
                f"expected at least {neqs + 1}."
            )

        y60 = data[:neqs]
        N_saved = data[neqs]

        obs = flow_observables_from_y(y60)

        rows.append({
            "accepted_index": model["accepted_index"],

            # ------------------------
            # N check
            # ------------------------
            "N_esigma": N_saved,

            # ------------------------
            # Saved summary values
            # ------------------------
            "r_summary": model["r"],
            "ns_summary": model["n_s"],
            "alpha_summary": model["alpha_s"],

            # ------------------------
            # Reconstructed from y60
            # ------------------------
            "r_from_esigma": obs["r_from_esigma"],
            "ns_from_esigma": obs["ns_from_esigma"],
            "alpha_from_esigma": obs["alpha_s_from_esigma"],

            # ------------------------
            # Differences
            # ------------------------
            "delta_r": (
                obs["r_from_esigma"]
                - model["r"]
            ),

            "delta_ns": (
                obs["ns_from_esigma"]
                - model["n_s"]
            ),

            "delta_alpha": (
                obs["alpha_s_from_esigma"]
                - model["alpha_s"]
            ),

            # ------------------------
            # Evolved N=60 hierarchy
            # ------------------------
            "eps_N60": obs["eps_N60"],
            "sig_N60": obs["sig_N60"],
            "lam2_N60": obs["lam2_N60"],
            "lam3_N60": obs["lam3_N60"],
            "lam4_N60": obs["lam4_N60"],
            "lam5_N60": obs["lam5_N60"],

            # ------------------------
            # Original proposal values
            # ------------------------
            "eps_input": model["eps"],
            "sig_input": model["sig"],
            "lam2_input": model["lam2"],
            "lam3_input": model["lam3"],
            "lam4_input": model["lam4"],
            "lam5_input": model["lam5"],

            "outdir": outdir,
        })

    result = pd.DataFrame(rows)

    return result

In [22]:
summary_file = (
    "/Users/epmeador/Desktop/research/rwarthur/"
    "inflation_gravitywaves/inflation_code/"
    "Slow-Roll Parameters Tests/higgs_potential_tests/"
    "neqs8_higgs_batch001/neqs8_summary.csv"
)

check_df = check_esigma_against_summary(
    summary_file,
    neqs=8,
)

In [23]:
print(
    check_df[
        [
            "accepted_index",
            "N_esigma",
            "r_summary",
            "r_from_esigma",
            "delta_r",
            "ns_summary",
            "ns_from_esigma",
            "delta_ns",
            "alpha_summary",
            "alpha_from_esigma",
            "delta_alpha",
        ]
    ].to_string(index=False)
)

 accepted_index  N_esigma  r_summary  r_from_esigma   delta_r  ns_summary  ns_from_esigma  delta_ns  alpha_summary  alpha_from_esigma  delta_alpha
              1      60.0   0.003209       0.002737 -0.000472    0.966934        0.969485  0.002552      -0.000553          -0.000470     0.000083
              2      60.0   0.002717       0.002337 -0.000380    0.969037        0.970297  0.001260      -0.000324          -0.000185     0.000140


In [24]:
print("\nMAXIMUM ABSOLUTE DIFFERENCES")
print(
    "max |Δr|       =",
    np.max(np.abs(check_df["delta_r"]))
)

print(
    "max |Δns|      =",
    np.max(np.abs(check_df["delta_ns"]))
)

print(
    "max |Δalpha_s| =",
    np.max(np.abs(check_df["delta_alpha"]))
)

print(
    "N range        =",
    check_df["N_esigma"].min(),
    "to",
    check_df["N_esigma"].max(),
)


MAXIMUM ABSOLUTE DIFFERENCES
max |Δr|       = 0.0004715628053604663
max |Δns|      = 0.002551514151541201
max |Δalpha_s| = 0.0001395904632934106
N range        = 60.0 to 60.0


These differences are fucking tiny good, that means test_esigma corresponds to the y state vector at 60 and those are the r, ns, and alphas I compute. Great teh code agrees with itself.

The fancy way of saying it:
The state stored in test_esigma_neqs8.dat is the evolved Hubble-flow state at N=60, and evaluating the analytic Hubble-flow expressions for r, ns, alphas on that state reproduces the observables used by the driver for CMB acceptance.


NOW does the pivot scale I am using correspond to the N=60 point? Or rather, does my N=60 CMB acceptance correspond to CMB-consistent observables in the full numerical spectrum at the physical k=0.05 pivot?

If we wrap around one accepted model from the summary:

In [38]:
import os
import numpy as np
import pandas as pd

NEQS = 8

summary_file = (
    "/Users/epmeador/Desktop/research/rwarthur/"
    "inflation_gravitywaves/inflation_code/"
    "Slow-Roll Parameters Tests/higgs_potential_tests/"
    "neqs8_higgs_batch001/neqs8_summary.csv"
)

# Load accepted models
summary = pd.read_csv(summary_file)

# Pick one accepted model
row = summary.iloc[0]

folder = row["outdir"]

# Load its spectra
s = np.loadtxt(
    os.path.join(folder, f"spec_s_neqs{NEQS}.dat")
)

t = np.loadtxt(
    os.path.join(folder, f"spec_t_neqs{NEQS}.dat")
)

# Unpack
k_s = s[:, 0]
Ps = s[:, 1]

k_t = t[:, 0]
Pt = t[:, 1]

# Saved flow observables
r_flow = row["r"]
ns_flow = row["n_s"]
alpha_flow = row["alpha_s"]

print("folder =", folder)
print("r_flow =", r_flow)
print("ns_flow =", ns_flow)
print("alpha_flow =", alpha_flow)
print("scalar points =", len(k_s))
print("tensor points =", len(k_t))

folder = /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8_higgs_batch001/lam2_2.7897200000e-04_lam3_-4.6097100000e-06_lam4_6.8706500000e-08_lam5_-8.9246100000e-09
r_flow = 0.0032086723332278
ns_flow = 0.9669338806724068
alpha_flow = -0.0005534951913177
scalar points = 41
tensor points = 41


In [39]:
k_pivot = 0.05
# k_pivot = 3.2e-4

# Find closest saved scalar/tensor k values
i_s = np.argmin(np.abs(k_s - k_pivot))
i_t = np.argmin(np.abs(k_t - k_pivot))

# Pull out the corresponding powers
k_s_pivot = k_s[i_s]
Ps_pivot = Ps[i_s]

k_t_pivot = k_t[i_t]
Pt_pivot = Pt[i_t]

# Compute r from the numerical spectra
r_spec = Pt_pivot / Ps_pivot

print("Requested k* =", k_pivot)
print("Closest scalar k =", k_s_pivot)
print("Closest tensor k =", k_t_pivot)

print("Ps(k*) =", Ps_pivot)
print("Pt(k*) =", Pt_pivot)

print("r_flow =", r_flow)
print("r_spec =", r_spec)
print("delta_r =", r_spec - r_flow)





Requested k* = 0.05
Closest scalar k = 0.05
Closest tensor k = 0.05
Ps(k*) = 2.0734199967002724e-09
Pt(k*) = 6.904903299727694e-12
r_flow = 0.0032086723332278
r_spec = 0.0033302000128852074
delta_r = 0.00012152767965740736


In [32]:
pct_diff = 100 * (r_spec - r_flow) / r_flow
print(pct_diff)

-13.727044530595496


In [33]:
esigma = np.loadtxt(
    os.path.join(folder, f"test_esigma_neqs{NEQS}.dat")
)

eps_60 = esigma[2]
sig_60 = esigma[3]

r_first = 16.0 * eps_60

c = 0.0814514
r_second = (
    16.0
    * eps_60
    * (1.0 - c * (sig_60 + 2.0 * eps_60))
)

print("epsilon_N60 =", eps_60)
print()
print("16 epsilon first order r  =", r_first)
print("second-order r flow =", r_second)
print("saved r flow     =", r_flow)
print("numerical r spec =", r_spec)

epsilon_N60 = 0.0001706453

16 epsilon first order r  = 0.0027303248
second-order r flow = 0.0027371095278673338
saved r flow     = 0.0032086723332278
numerical r spec = 0.0027682164532047224


The spectrum code anchors to 

        Ninit = N
        spec_params.a_init = (1.73e-61/y[1]) * np.exp(Ninit)
        
Ninit = 60 and  a(Ninit) = a_init* e^(-Ninit)

If a0 = 1, then k0 = a0H0 = H0 where H0= size of present day comoving hubble wavenumber in planck units

The code is normalizing the inflationary scale factor so that the comoving Hubble wavenumber k=aH at the chosen initial point equals a particular present-day comoving scale expressed in Planck units.

So, at horizon crossing k=aH, we are forcing at Ninit it must be true that aH=1.73e-61


So what k does that scale correspond to?

Given that we have k_planck = k_mpc * 5.41e-58 or k_planck = k_physical * 5.41e-58 so 

k_physical = k_planck/5.41e-58, when k_planck = aH (at Ninit) = 1.73e-61 you get that
k_physical = 1.73e-61/5.41e-58 = 3.2e-4 mpc^-1.

H0 sets the present-day Hubble wavenumber when a0=1

---------------------------------------------------------------------------------------------------------------


Better:
Okay so physically during inflation every mode has a horizon crossing time, that occurs at k(N)=a(N)H(N), which means at some N the mode is crossing outside the horizon. The code chooses a normalization such that a(Ninit) H(Ninit) = 1.73e-61. Given Ninit = 60, we are saying that at N=60 (60 efolds before the end of inflation, the mode with k = 1.73e-61 Mpl is crossing the horizon. The present day mode that corresponds to that is derived from k_physical = 1.73e-61/5.41e-58 = 3.2e-4 mpc^-1. So N=60 corresponds to k of 3.2e-4 mpc^-1.

So I have been evaluating observables at N=60, but then comparing the numerical spectra I get at 0.05 mpc^-1. 0.05 is much larger than 3.2e-4 . Specifically, kpivot/kactual= 0.05/3.2e-4 ~ 156. In reality at k=0.05 hmpc^1, the value of N should be smaller (k is larger corresponds to smaller wavelength). So when I compare the r value I get from the spectra and the flow at N=60, it makes sense that they would be different givem this discrepancy.

Remember that, k ~ e^(N), so ln k = N. This difference in N from Ninit, would be ln (kpivot/kactual) = ΔN. We then have ln (0.05/3.2e-4) = 5.05 = ΔN
N: 60 - 5.05  = 54.94854, so that N ~ 54.95 should correspond to k= 0.05 hmpc^-1

In [34]:
# load the saved path for this same model
path_file = [
    f for f in os.listdir(folder)
    if f.startswith(f"path_neqs{NEQS}_")
][0]

path = np.loadtxt(
    os.path.join(folder, path_file)
)

# Columns:
# 0 phi
# 1 H
# 2 epsilon
# 3 sigma
# 4 lambda2
# 5 lambda3
# 6 lambda4
# 7 lambda5
# 8 N
N_path = path[:, NEQS]

# Find nearest point to N = 55
i55 = np.argmin(np.abs(N_path - 55.0))

N_55 = N_path[i55]

eps_55 = path[i55, 2]
sig_55 = path[i55, 3]

c = 0.0814514

r_55 = (
    16.0
    * eps_55
    * (1.0 - c * (sig_55 + 2.0 * eps_55))
)

print("Closest N =", N_55)
print("epsilon(N) =", eps_55)
print("sigma(N)   =", sig_55)

print("r_flow at N~55 =", r_55)
print("r_spec at k=0.05 =", r_spec)
print("difference =", r_spec - r_55)




Closest N = 56.035268
epsilon(N) = 0.0001933307
sigma(N)   = -0.03288321
r_flow at N~55 = 0.003101478799622314
r_spec at k=0.05 = 0.0027682164532047224
difference = -0.00033326234641759163


In [16]:
mask = (N_path > 50) & (N_path < 60)
print(np.sort(N_path[mask]))

[52.8302   56.035268 59.430355]


In [18]:
from scipy.interpolate import CubicSpline

order = np.argsort(N_path)
N_sorted = N_path[order]
eps_sorted = path[order, 2]
sig_sorted = path[order, 3]

eps_spline = CubicSpline(N_sorted, eps_sorted)
sig_spline = CubicSpline(N_sorted, sig_sorted)

eps_55_interp = eps_spline(55.0)
sig_55_interp = sig_spline(55.0)

r_55_interp = 16.0 * eps_55_interp * (1.0 - c*(sig_55_interp + 2.0*eps_55_interp))
print("interpolated eps(55) =", eps_55_interp)
print("interpolated r(55)   =", r_55_interp)

interpolated eps(55) = 0.00020000307587751063
interpolated r(55)   = 0.003208666110105664


In [17]:
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq
import numpy as np

# -----------------------------------
# Pull background quantities
# -----------------------------------

N_path = path[:, NEQS]
H_path = path[:, 1]
eps_path = path[:, 2]
sig_path = path[:, 3]

# Make sure N is increasing
order = np.argsort(N_path)

N_path = N_path[order]
H_path = H_path[order]
eps_path = eps_path[order]
sig_path = sig_path[order]

# -----------------------------------
# Interpolate the background
# -----------------------------------

H_of_N = PchipInterpolator(N_path, H_path)
eps_of_N = PchipInterpolator(N_path, eps_path)
sig_of_N = PchipInterpolator(N_path, sig_path)

# -----------------------------------
# Anchor
# -----------------------------------

N_anchor = 60.0
k_anchor = 1.73e-61 / 5.41e-58

H_anchor = H_of_N(N_anchor)

# k(N) from horizon crossing k = aH
def k_of_N(N):
    return (
        k_anchor
        * np.exp(N_anchor - N)
        * H_of_N(N) / H_anchor
    )

# -----------------------------------
# Solve k(N) = 0.05
# -----------------------------------

k_pivot = 0.05

def root_function(N):
    return np.log(k_of_N(N) / k_pivot)

N_pivot = brentq(
    root_function,
    N_path.min(),
    N_path.max(),
)

print("N corresponding to k=0.05 =", N_pivot)
print("k(N_pivot) =", k_of_N(N_pivot))

eps_pivot = float(eps_of_N(N_pivot))
sig_pivot = float(sig_of_N(N_pivot))

c = 0.0814514

r_flow_pivot = (
    16.0
    * eps_pivot
    * (1.0 - c*(sig_pivot + 2.0*eps_pivot))
)

print()
print("epsilon at pivot =", eps_pivot)
print("sigma at pivot   =", sig_pivot)

print()
print("r_flow at k=0.05 =", r_flow_pivot)
print("r_spec at k=0.05 =", r_spec)
print("difference       =", r_spec - r_flow_pivot)

N corresponding to k=0.05 = 54.94691510680546
k(N_pivot) = 0.04999999999999941

epsilon at pivot = 0.00020035235525985582
sigma at pivot   = -0.03348942392624824

r_flow at k=0.05 = 0.003214277270420988
r_spec at k=0.05 = 0.0033302000128852074
difference       = 0.00011592274246421952


So, now we maybe want to change the observables so i am evaulating them directly at N=55 and the models that pass that are accepted??


If I do the following in my acceptance cuts this ought to roughly work. But then i need to make sure the r that I am giving my code is one to be observed at the pivot scale k=0.05 hMpc^-1. The key is that the target values should come from the CMB constraints at that pivot, not from Higgs base model evaluated at N=60. We want the the observationally allowed r range at the chosen CMB pivot.

The way my current code is set up is like asking Which models have the desired sobervables at k≃3.2×10−4? I need to get r and ns at the k pivot, and then I will input those into my code and find models that pass that at the N that corresponds to the kpivot scale.

In [1]:
mask = (N_path > 50) & (N_path < 60)
print(np.sort(N_path[mask]))

NameError: name 'N_path' is not defined